In [0]:
# =============================================================================
# CONFIGURAÇÕES E UTILITÁRIOS DE GRAVAÇÃO
# =============================================================================

import json
import re
import pandas as pd
import pyarrow as pa
from datetime import datetime, timezone
from deltalake import DeltaTable
from deltalake.writer import write_deltalake

# Constantes locais
CONTAINER = "squad1"

def get_delta_path(camada: str, tabela: str, storage_opts: dict) -> str:
    """Usa a URI nativa abfss:// com tratamento para gravação na raiz (camada vazia)."""
    conta = storage_opts.get("account_name", "internshipdatalake")

    # Monta só o corpo do caminho (sem o protocolo), evitando barras duplicadas
    corpo = tabela if not camada else f"{camada}/{tabela}"
    corpo = corpo.strip("/")
    corpo = re.sub(r"/+", "/", corpo)  # colapsa // apenas dentro do corpo, nunca no protocolo

    return f"abfss://{CONTAINER}@{conta}.dfs.core.windows.net/{corpo}"

def delta_existe(camada: str, tabela: str, storage_opts: dict) -> bool:
    try:
        # Traduz dinamicamente os parâmetros para o formato esperado pelo delta-rs SDK
        opts_sdk = {
            "account_name": storage_opts.get("account_name", "internshipdatalake"),
            "client_id": storage_opts.get("client_id"),
            "client_secret": storage_opts.get("client_secret"),
            "tenant_id": storage_opts.get("tenant_id")
        }
        DeltaTable(get_delta_path(camada, tabela, storage_opts), storage_options=opts_sdk)
        return True
    except Exception:
        return False

def gravar_delta(df, camada: str, tabela: str, storage_opts: dict, mode: str = "append", particionar: bool = True) -> bool:
    path = get_delta_path(camada, tabela, storage_opts)
    modo_real = mode if delta_existe(camada, tabela, storage_opts) else "overwrite"

    try:
        # 1. Conversão para Pandas
        pdf = df.toPandas()

        # 2. Correção OBRIGATÓRIA de fuso horário (Evita erro fatal no PyArrow)
        for col_name in pdf.columns:
            if pd.api.types.is_datetime64_any_dtype(pdf[col_name]):
                pdf[col_name] = pdf[col_name].dt.tz_localize(None)

        # 3. Conversão para Tabela Arrow
        tabela_arrow = pa.Table.from_pandas(pdf, preserve_index=False)

        # 4. Definição Dinâmica de Partições
        partition_by = None
        if particionar and camada == "bronze":
            possiveis = ["ano", "mes", "dia", "hora"]
            colunas_pdf = pdf.columns.tolist()
            partition_by = [c for c in possiveis if c in colunas_pdf]
            if not partition_by: 
                partition_by = None

        # 5. Normalização do dicionário para o SDK do delta-rs
        opts_sdk = {
            "account_name": storage_opts.get("account_name", "internshipdatalake"),
            "client_id": storage_opts.get("client_id"),
            "client_secret": storage_opts.get("client_secret"),
            "tenant_id": storage_opts.get("tenant_id")
        }

        # 6. Gravação física direta (Bypass do Databricks Serverless)
        write_deltalake(
            table_or_uri=path,
            data=tabela_arrow,
            mode=modo_real,
            storage_options=opts_sdk,
            partition_by=partition_by,
            schema_mode="overwrite" if modo_real == "overwrite" else "merge"
        )
        
        print(f"[Sucesso] Gravado fisicamente via SDK em: {path} | Linhas: {len(pdf)}")
        return True

    except Exception as e:
        print(f"[Erro] Falha ao gravar {path}: {str(e)}")
        return False

In [0]:
# ==============================================================================
# CAMADA BRONZE
# ==============================================================================
import json
from datetime import datetime, timezone

# ==============================================================================
# FUNÇÕES AUXILIARES DE ADLS E CONTROLE (JSON) 
# ==============================================================================

def garantir_diretorio(container_client, caminho):
    """Cria diretório no ADLS se ainda não existir."""
    try:
        container_client.create_directory(caminho)
        print(f"Diretório criado: {caminho}")
    except Exception:
        pass # Diretório já existe

def ler_json_adls(container_client, caminho_arquivo, padrao=None):
    """Lê JSON no ADLS. Se não existir, retorna o padrão."""
    if padrao is None:
        padrao = []
    try:
        file_client = container_client.get_file_client(caminho_arquivo)
        conteudo = file_client.download_file().readall().decode("utf-8")
        if not conteudo.strip():
            return padrao
        return json.loads(conteudo)
    except Exception:
        return padrao

def salvar_json_adls(container_client, caminho_arquivo, dados):
    """Salva JSON no ADLS, criando diretórios pais se necessário."""
    pasta_pai = "/".join(caminho_arquivo.split("/")[:-1])
    garantir_diretorio(container_client, pasta_pai)

    file_client = container_client.get_file_client(caminho_arquivo)
    conteudo = json.dumps(dados, ensure_ascii=False, indent=2)
    file_client.upload_data(conteudo, overwrite=True)
    print(f"JSON de controle salvo em: {caminho_arquivo}")

def listar_controles_entidade(container_client, pasta_controle_base):
    """Lista todos os arquivos controle_leitura.json de uma entidade específica."""
    controles = []
    try:
        for item in container_client.get_paths(path=pasta_controle_base, recursive=True):
            if item.name.endswith("controle_leitura.json"):
                controles.append(item.name)
    except Exception:
        controles = []
    return controles

def carregar_arquivos_ja_lidos(container_client, pasta_controle_base):
    """Carrega arquivos já lidos considerando todos os controles existentes da entidade."""
    arquivos_lidos = set()
    controles = listar_controles_entidade(container_client, pasta_controle_base)

    for caminho_controle in controles:
        registros = ler_json_adls(container_client, caminho_controle, padrao=[])
        for r in registros:
            arquivo = r.get("arquivo_origem")
            status = r.get("status")
            if arquivo and status == "LIDO":
                arquivos_lidos.add(arquivo)

    return arquivos_lidos, controles

def extrair_data_do_caminho_raw(caminho):
    """Extrai data do padrão real-time-data/YYYY/MM/DD/HHMMSS/arquivo.parquet."""
    try:
        partes = caminho.split("/")
        ano, mes, dia = int(partes[1]), int(partes[2]), int(partes[3])
        hora_min_seg = partes[4]
        hora = int(hora_min_seg[0:2])
        minuto = int(hora_min_seg[2:4])
        segundo = int(hora_min_seg[4:6])
        return datetime(ano, mes, dia, hora, minuto, segundo, tzinfo=timezone.utc)
    except Exception:
        return datetime.min.replace(tzinfo=timezone.utc)

In [0]:
# ==============================================================================
# CAMADA SILVER
# ==============================================================================
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

def ler_delta(camada, tabela, storage_opts):
    import pyspark.sql.functions as F
    
    # 1. Montagem dinâmica do caminho base
    if camada == "":
        caminho_base = f"squad1@internshipdatalake.dfs.core.windows.net/{tabela}"
    else:
        caminho_base = f"squad1@internshipdatalake.dfs.core.windows.net/{camada}/{tabela}"
        
    # Limpa barras duplicadas apenas no corpo do caminho, preservando o protocolo intacto
    caminho_limpo = caminho_base.replace("//", "/")
    caminho_final = f"abfss://{caminho_limpo}"
    
    # 2. Desempacota STORAGE_OPTIONS e mapeia as chaves OAuth do Hadoop para o Spark Reader
    account_name = storage_opts.get("account_name", "internshipdatalake")
    client_id = storage_opts.get("client_id")
    client_secret = storage_opts.get("client_secret")
    tenant_id = storage_opts.get("tenant_id")
    
    return (spark.read
        .format("delta")
        .option(f"fs.azure.account.auth.type.{account_name}.dfs.core.windows.net", "OAuth")
        .option(f"fs.azure.account.oauth.provider.type.{account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
        .option(f"fs.azure.account.oauth2.client.id.{account_name}.dfs.core.windows.net", client_id)
        .option(f"fs.azure.account.oauth2.client.secret.{account_name}.dfs.core.windows.net", client_secret)
        .option(f"fs.azure.account.oauth2.client.endpoint.{account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
        .option("ignoreCorruptFiles", "true")
        .option("ignoreMissingFiles", "true")
        .load(caminho_final))
        
def schema_dq_logs():
    """
    Retorna o schema padronizado para a tabela de logs de qualidade de dados (DQ).
    """
    return StructType([
        StructField("run_id", StringType(), True),
        StructField("tabela", StringType(), True),
        StructField("regra", StringType(), True),
        StructField("status", StringType(), True),
        StructField("severidade", StringType(), True),
        StructField("qtd_registros_falhos", IntegerType(), True),
        StructField("qtd_registros_total", IntegerType(), True),
        StructField("timestamp_execucao", TimestampType(), True),
        StructField("arquivo_origem", StringType(), True)
    ])

In [0]:
# ==============================================================================
# REFERÊNCIA CRUZADA COM FALLBACK (Silver -> Bronze)
# ==============================================================================
def obter_referencia_silver_ou_bronze(tabela: str, colunas: list, storage_opts: dict = None):
    """
    Versão genérica: aceita uma lista de colunas (não só um único ID), útil
    quando a regra precisa de mais de uma coluna da tabela de referência
    (ex: id_pedido + valor_total, ou id_categoria + id_categoria_pai).
    Prioriza a Silver (dado já validado) e cai para a Bronze como fallback
    quando a Silver ainda não existir.
    """
    opts = storage_opts if storage_opts is not None else STORAGE_OPTIONS

    if delta_existe("silver", tabela, opts):
        print(f"Referência de {tabela}: usando Silver (dado já validado).")
        return ler_delta("silver", tabela, opts).select(*colunas).dropDuplicates()
    elif delta_existe("bronze", tabela, opts):
        print(f"Referência de {tabela}: Silver ainda não existe — usando Bronze como fallback.")
        return ler_delta("bronze", tabela, opts).select(*colunas).dropDuplicates()
    else:
        print(f"Referência de {tabela}: nem Silver nem Bronze encontradas.")
        schema = StructType([StructField(c, StringType(), True) for c in colunas])
        return spark.createDataFrame([], schema)


def obter_referencia_ids_silver_ou_bronze(tabela: str, coluna_id: str, storage_opts: dict = None):
    """
    Atalho de coluna única sobre obter_referencia_silver_ou_bronze — usado
    nas checagens simples de FK/associação (ex: "esse id_cliente já tem
    pedido?").
    """
    return obter_referencia_silver_ou_bronze(tabela, [coluna_id], storage_opts)


def obter_referencia_ids_uniao_silver_bronze(tabela: str, coluna_id: str, storage_opts: dict = None):
    """
    Variante para checagens de EXISTÊNCIA de FK (ex: "esse id_cliente
    existe em algum lugar?"). Diferente de obter_referencia_ids_silver_ou_bronze
    (que escolhe Silver OU Bronze, priorizando Silver quando ela existe),
    esta função UNE os IDs das duas camadas — porque, uma vez que a Silver
    já existe, um registro pode legitimamente existir na Bronze e ainda não
    ter migrado para a Silver (ex: preso na própria quarentena daquela
    tabela por outro motivo). Usar só "Silver OU Bronze" faria esse
    registro ser tratado como inexistente/órfão por engano, quando na
    verdade ele existe — só não foi validado ainda.
    """
    opts = storage_opts if storage_opts is not None else STORAGE_OPTIONS

    ids_silver = None
    if delta_existe("silver", tabela, opts):
        ids_silver = ler_delta("silver", tabela, opts).select(coluna_id).dropDuplicates()

    ids_bronze = None
    if delta_existe("bronze", tabela, opts):
        ids_bronze = ler_delta("bronze", tabela, opts).select(coluna_id).dropDuplicates()

    if ids_silver is not None and ids_bronze is not None:
        print(f"Referência de {tabela}: unindo IDs de Silver + Bronze.")
        return ids_silver.unionByName(ids_bronze).dropDuplicates()
    elif ids_silver is not None:
        return ids_silver
    elif ids_bronze is not None:
        return ids_bronze
    else:
        print(f"Referência de {tabela}: nem Silver nem Bronze encontradas.")
        schema = StructType([StructField(coluna_id, LongType(), True)])
        return spark.createDataFrame([], schema)